In [ ]:
"""
figure.py

This script generates figures based on the cleaned modelling dataset and SDM impact results.

Main outputs:
1. Violin plots across urbanity levels for energy, landscape, and socio-demographic variables.
2. SDM impact plots showing direct, indirect, and total effects.

Required input files:
- model/form_energy_final_cleaned_renamed.csv
- output/sdm_impacts_with_p.xlsx

Recommended project structure:
project/
├── data/
├── output/
├── model/
├── figures/
└── figure.py
"""

import os
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)


# ============================================================
# 0. PROJECT STRUCTURE
# ============================================================

BASE_DIR = os.getcwd()

DATA_DIR = os.path.join(BASE_DIR, "data")
OUT_DIR = os.path.join(BASE_DIR, "output")
WEIGHT_DIR = os.path.join(BASE_DIR, "weights")
SRC_DIR = os.path.join(BASE_DIR, "src")
SHP_DIR = os.path.join(BASE_DIR, "shp")
MODEL_DIR = os.path.join(BASE_DIR, "model")
FIGURE_DIR = os.path.join(BASE_DIR, "figures")

for directory in [DATA_DIR, OUT_DIR, WEIGHT_DIR, SRC_DIR, SHP_DIR, MODEL_DIR, FIGURE_DIR]:
    os.makedirs(directory, exist_ok=True)

print("PROJECT STRUCTURE READY")
print("BASE_DIR:", BASE_DIR)


# ============================================================
# 1. GLOBAL FIGURE SETTINGS
# ============================================================

FONT_BASE = 14
FIGSIZE_VIOLIN = (12, 6)

FONT = {
    "title": FONT_BASE * 1.4,
    "label": FONT_BASE * 1.2,
    "tick": FONT_BASE * 1.1,
    "anno": FONT_BASE * 0.9
}

VAR_NAME_MAP = {
    "ec_total": "REC",
    "ec_inten_percapita": "REPC",
    "AI_Dense_Short": "AI Dense Short",
    "PLAND_Dense_Short": "PLAND Dense Short",
    "AI_Dense_Tall": "AI Dense Tall",
    "PLAND_Dense_Tall": "PLAND Dense Tall",
    "ave_floor": "Average Floor Area",
    "pop_density": "Population Density",
    "hh_size": "Household Size"
}


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def safe_filename(name: str) -> str:
    """Convert variable or figure names into safe filenames."""
    name = str(name).strip()
    name = re.sub(r"[^A-Za-z0-9_\-]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")


def remove_outliers_iqr(group: pd.Series) -> pd.Series:
    """Remove outliers using the 1.5 IQR rule within one group."""
    q1, q3 = group.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return group.where((group > lower) & (group < upper))


def parse_effect_column(col: pd.Series) -> tuple[pd.Series, pd.Series]:
    """
    Convert values like '0.012**' into numeric effects and significance flags.

    Returns:
    - numeric effect values
    - boolean significance indicator
    """
    sig = col.astype(str).str.contains(r"\*")
    numeric = (
        col.astype(str)
        .str.replace("*", "", regex=False)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
        .astype(float)
    )
    return numeric, sig


# ============================================================
# 3. URBANITY VIOLIN PLOTS
# ============================================================

def plot_urbanity_violin(
    df: pd.DataFrame,
    target_var: str,
    urbanity_col: str = "ste_mvs",
    scale: float = 1.0,
    unit_label: str = "",
    output_dir: str = FIGURE_DIR,
    show_plot: bool = False
) -> None:
    """Generate and save a violin plot across urbanity levels."""

    if urbanity_col not in df.columns:
        print(f"⚠️ Skipped {target_var}: urbanity column '{urbanity_col}' not found.")
        return

    if target_var not in df.columns:
        print(f"⚠️ Skipped {target_var}: variable not found in dataframe.")
        return

    print(f"\n📌 Generating violin plot for: {target_var}")

    sns.set_style("whitegrid")
    fig, ax = plt.subplots(figsize=FIGSIZE_VIOLIN)

    df_clean = df[[urbanity_col, target_var]].copy()
    df_clean[urbanity_col] = pd.to_numeric(df_clean[urbanity_col], errors="coerce")
    df_clean[target_var] = pd.to_numeric(df_clean[target_var], errors="coerce")

    df_clean = df_clean.dropna()
    df_clean = df_clean[df_clean[urbanity_col].isin([1, 2, 3, 4, 5])]
    df_clean = df_clean[df_clean[target_var] > 0]

    if df_clean.empty:
        print(f"⚠️ Skipped {target_var}: no valid data after cleaning.")
        plt.close(fig)
        return

    df_clean[target_var] = (
        df_clean.groupby(urbanity_col)[target_var]
        .transform(remove_outliers_iqr)
    )
    df_clean = df_clean.dropna()

    if df_clean.empty:
        print(f"⚠️ Skipped {target_var}: no valid data after outlier removal.")
        plt.close(fig)
        return

    df_clean[target_var] = df_clean[target_var] / scale

    medians = df_clean.groupby(urbanity_col)[target_var].median()
    print("Urbanity median values:")
    print(medians)

    sns.violinplot(
        data=df_clean,
        x=urbanity_col,
        y=target_var,
        palette="coolwarm",
        inner="box",
        bw=0.35,
        cut=0.05,
        gridsize=300,
        scale="width",
        linewidth=0.8,
        ax=ax
    )

    ax.set_xticks([0, 1, 2, 3, 4])
    ax.set_xticklabels(
        ["Urbanity 1", "Urbanity 2", "Urbanity 3", "Urbanity 4", "Urbanity 5"],
        fontsize=FONT["tick"]
    )

    ymax = df_clean[target_var].max()
    ax.set_ylim(0, ymax * 1.1)
    ax.tick_params(axis="y", labelsize=FONT["tick"])

    label_name = VAR_NAME_MAP.get(target_var, target_var)
    ax.set_xlabel("Urbanity Levels", fontsize=FONT["label"], labelpad=12)

    if unit_label:
        ax.set_ylabel(f"{label_name} ({unit_label})", fontsize=FONT["label"], labelpad=9)
    else:
        ax.set_ylabel(label_name, fontsize=FONT["label"], labelpad=9)

    ax.set_title(
        f"Distribution of {label_name} across Urbanity Levels",
        fontsize=FONT["title"],
        pad=12
    )

    ax.text(-0.5, -ymax * 0.10, "(Urban)", ha="center", fontsize=FONT["label"], weight="bold")
    ax.text(4.5, -ymax * 0.10, "(Rural)", ha="center", fontsize=FONT["label"], weight="bold")

    fig.tight_layout()

    output_path = os.path.join(output_dir, f"violin_urbanity_{safe_filename(target_var)}.png")
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"✅ Saved violin plot: {output_path}")

    if show_plot:
        plt.show()
    else:
        plt.close(fig)


def generate_all_urbanity_violin_plots(show_plot: bool = False) -> None:
    """Generate all urbanity-level violin plots used in the analysis."""

    input_file = os.path.join(MODEL_DIR, "form_energy_final_cleaned_renamed.csv")

    if not os.path.exists(input_file):
        print(f"⚠️ Model dataset not found. Skipped violin plots: {input_file}")
        return

    df = pd.read_csv(input_file)

    violin_specs = [
        # Energy variables
        {"target_var": "ec_total", "scale": 1e6, "unit_label": "GWh"},
        {"target_var": "ec_inten_percapita", "scale": 1e3, "unit_label": "MWh / capita"},

        # Landscape variables
        {"target_var": "AI_Dense_Short", "scale": 1.0, "unit_label": ""},
        {"target_var": "PLAND_Dense_Short", "scale": 1.0, "unit_label": "%"},
        {"target_var": "AI_Dense_Tall", "scale": 1.0, "unit_label": ""},
        {"target_var": "PLAND_Dense_Tall", "scale": 1.0, "unit_label": "%"},

        # Socio-demographic and building variables
        {"target_var": "ave_floor", "scale": 1.0, "unit_label": ""},
        {"target_var": "pop_density", "scale": 1.0, "unit_label": ""},
        {"target_var": "hh_size", "scale": 1.0, "unit_label": ""},
    ]

    for spec in violin_specs:
        plot_urbanity_violin(
            df=df,
            target_var=spec["target_var"],
            scale=spec["scale"],
            unit_label=spec["unit_label"],
            show_plot=show_plot
        )


# ============================================================
# 4. SDM IMPACT PLOTS
# ============================================================

COLOR_DIRECT = "#e29a7e"
COLOR_INDIRECT = "#9ecae9"


def plot_impact_group(df_group: pd.DataFrame, title: str, figsize: tuple[int, int], output_path: str, show_plot: bool = False) -> None:
    """Plot direct, indirect, and total SDM effects for one variable group."""

    if df_group.empty:
        print(f"⚠️ Skipped impact plot '{title}': no data available.")
        return

    fig, ax = plt.subplots(figsize=figsize)
    ypos = np.arange(0, len(df_group))
    bar_h = 0.8

    for i, row in df_group.iterrows():
        y = ypos[i]

        direct = row["direct"]
        indirect = row["indirect"]
        total = row["total"]

        alpha_direct = 1.0 if row["direct_sig"] else 0.3
        alpha_indirect = 1.0 if row["indirect_sig"] else 0.3

        # Positive stacking
        left = 0
        if direct > 0:
            ax.barh(y, direct, left=left, color=COLOR_DIRECT, alpha=alpha_direct, height=bar_h)
            left += direct
        if indirect > 0:
            ax.barh(y, indirect, left=left, color=COLOR_INDIRECT, alpha=alpha_indirect, height=bar_h)

        # Negative stacking
        left = 0
        if direct < 0:
            ax.barh(y, direct, left=left, color=COLOR_DIRECT, alpha=alpha_direct, height=bar_h)
            left += direct
        if indirect < 0:
            ax.barh(y, indirect, left=left, color=COLOR_INDIRECT, alpha=alpha_indirect, height=bar_h)

        # Hatch for significant total effect
        if row["total_sig"] and total != 0:
            if total > 0:
                left = 0
                width = total
            else:
                left = total
                width = abs(total)

            ax.barh(
                y,
                width,
                left=left,
                facecolor="none",
                edgecolor="grey",
                hatch="///",
                zorder=3,
                linewidth=0.0,
                height=bar_h
            )

    ax.set_yticks(ypos)
    ax.set_yticklabels(df_group["variable"], fontsize=12)
    ax.invert_yaxis()
    ax.axvline(0, color="black", linewidth=1)
    ax.grid(True, linestyle="--", alpha=0.6)

    ax.set_xlabel("Effect size", fontsize=12)
    ax.set_title(title, fontsize=16)

    handles = [
        plt.Rectangle((0, 0), 1, 1, color=COLOR_DIRECT, alpha=1, label="Direct (significant)"),
        plt.Rectangle((0, 0), 1, 1, color=COLOR_DIRECT, alpha=0.3, label="Direct (not significant)"),
        plt.Rectangle((0, 0), 1, 1, color=COLOR_INDIRECT, alpha=1, label="Indirect (significant)"),
        plt.Rectangle((0, 0), 1, 1, color=COLOR_INDIRECT, alpha=0.3, label="Indirect (not significant)"),
        plt.Rectangle((0, 0), 1, 1, facecolor="none", edgecolor="gray", hatch="///", label="Total (significant)")
    ]

    ax.legend(handles=handles)
    plt.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"✅ Saved SDM impact plot: {output_path}")

    if show_plot:
        plt.show()
    else:
        plt.close(fig)


def generate_sdm_impact_plots(show_plot: bool = False) -> None:
    """Generate SDM impact plots for socio-economic/building and landscape variables."""

    file_path = os.path.join(OUT_DIR, "sdm_impacts_with_p.xlsx")

    if not os.path.exists(file_path):
        print(f"⚠️ SDM impact file not found. Skipped SDM impact plots: {file_path}")
        return

    df = pd.read_excel(file_path)

    required_cols = ["variable", "direct_effect", "indirect_effect", "total_effect"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        print(f"⚠️ Missing columns in SDM impact file: {missing_cols}")
        return

    df = df.rename(columns={
        "direct_effect": "direct_raw",
        "indirect_effect": "indirect_raw",
        "total_effect": "total_raw"
    })

    df["direct"], df["direct_sig"] = parse_effect_column(df["direct_raw"])
    df["indirect"], df["indirect_sig"] = parse_effect_column(df["indirect_raw"])
    df["total"], df["total_sig"] = parse_effect_column(df["total_raw"])

    group1_vars = [
        "pop_density", "SF_pc", "inc_hi_pc",
        "hh_size", "pre2000_pc", "ave_floor", "Aplus_pc"
    ]

    group2_vars = [
        "PLAND_water", "AI_water", "IJI_water",
        "AI_built", "IJI_built",
        "PLAND_Dense_Short", "AI_Dense_Short",
        "PLAND_Dense_Tall", "AI_Dense_Tall",
        "PD", "SHDI"
    ]

    groups = {
        "SocioEconomic_Building": (group1_vars, (10, 5)),
        "LandscapeMetrics": (group2_vars, (10, 8))
    }

    for group_name, (var_list, figsize) in groups.items():
        existing_vars = [v for v in var_list if v in df["variable"].values]
        missing_vars = [v for v in var_list if v not in df["variable"].values]

        if missing_vars:
            print(f"⚠️ Missing variables in SDM impact file for {group_name}:")
            print(missing_vars)

        if not existing_vars:
            print(f"⚠️ No variables available for {group_name}. Skipped.")
            continue

        df_group = df[df["variable"].isin(existing_vars)].copy()
        df_group = df_group.set_index("variable").loc[existing_vars].reset_index()

        output_path = os.path.join(FIGURE_DIR, f"sdm_impacts_{safe_filename(group_name)}.png")
        plot_impact_group(
            df_group=df_group,
            title="SDM Impacts Measurement (KNN6)",
            figsize=figsize,
            output_path=output_path,
            show_plot=show_plot
        )


# ============================================================
# 5. RUN ALL FIGURES
# ============================================================

if __name__ == "__main__":
    generate_all_urbanity_violin_plots(show_plot=False)
    generate_sdm_impact_plots(show_plot=False)

    print("\n🎉 ALL FIGURES GENERATED SUCCESSFULLY!")
